# Representation Demystifies Flat Minima
## Notebook 29: Representation Matters

The previous notebooks established the repo's central sequence:

- **00** introduced flat and sharp minima.
- **13** explained Hessian geometry.
- **17** showed that the same function can have different measured sharpness.
- **23** showed how correlation can weaken under reparameterization.

This notebook synthesizes the lesson:

> Some properties belong to the computation. Others belong to the representation.

## Output files

Running this notebook creates:

- `figures/29_property_table.png`
- `figures/29_same_computation.png`
- `figures/29_geometry_changes.png`
- `figures/29_invariants.png`
- `results/29_representation_summary.csv`
- `results/29_representation_summary.json`
- `downloads/29_representation_matters_outputs.zip`

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. Two kinds of properties

A learned model has properties that may survive reparameterization and properties that may not.

The key distinction is:

- **computation-level properties**
- **representation-level properties**

In [ ]:
properties = pd.DataFrame([
    {
        "property": "Function values",
        "level": "computation",
        "survives_reparameterization": True,
        "reason": "Same input-output map"
    },
    {
        "property": "Predictions",
        "level": "computation",
        "survives_reparameterization": True,
        "reason": "Same outputs for same inputs"
    },
    {
        "property": "Test error",
        "level": "computation / evaluation",
        "survives_reparameterization": True,
        "reason": "Predictions are unchanged"
    },
    {
        "property": "Hessian",
        "level": "representation",
        "survives_reparameterization": False,
        "reason": "Curvature is coordinate-dependent"
    },
    {
        "property": "Sharpness",
        "level": "representation",
        "survives_reparameterization": False,
        "reason": "Measured through Hessian geometry"
    },
    {
        "property": "Parameter values",
        "level": "representation",
        "survives_reparameterization": False,
        "reason": "Coordinates change"
    },
    {
        "property": "Parameter norm",
        "level": "representation",
        "survives_reparameterization": False,
        "reason": "Scale changes under reparameterization"
    },
])

properties

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis("off")

display_df = properties[["property", "level", "survives_reparameterization", "reason"]].copy()
display_df["survives_reparameterization"] = display_df["survives_reparameterization"].map({True: "yes", False: "no"})

table = ax.table(
    cellText=display_df.values,
    colLabels=["Property", "Level", "Survives?", "Reason"],
    loc="center",
    cellLoc="left",
    colLoc="left"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.45)

ax.set_title("Which properties survive reparameterization?", pad=18)

fig_path = FIGURES / "29_property_table.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 2. Same computation

Use the simple model from Notebook 17:

\[
f(x; w) = wx
\]

with reparameterization:

\[
w = su
\]

At:

\[
u^* = \frac{1}{s}
\]

every representation computes the same function:

\[
f(x) = x
\]

In [ ]:
x = np.linspace(-1, 1, 400)
y = x
scales = np.array([0.25, 0.5, 1.0, 2.0, 4.0])

plt.figure(figsize=(8, 5))

for s in scales:
    u_star = 1 / s
    y_hat = (s * u_star) * x
    plt.plot(x, y_hat, label=f"s={s}, u*={u_star:.2f}")

plt.plot(x, y, linestyle="--", label="target y=x")
plt.xlabel("x")
plt.ylabel("prediction")
plt.title("Same computation under different representations")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "29_same_computation.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 3. Geometry changes

From Notebook 17:

\[
H_u = s^2 H_w
\]

The function remains the same, but measured curvature changes.

In [ ]:
ex2 = np.mean(x**2)
hessian_w = 2 * ex2

geometry = pd.DataFrame({
    "scale_s": scales,
    "u_star": 1 / scales,
    "effective_w": scales * (1 / scales),
    "hessian_u": 2 * (scales ** 2) * ex2,
    "hessian_w": hessian_w,
})

geometry["sharpness_ratio"] = geometry["hessian_u"] / geometry["hessian_w"]

geometry

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(geometry["scale_s"], geometry["hessian_u"], marker="o", label="measured Hessian in u")
plt.axhline(hessian_w, linestyle="--", label="Hessian in w")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("measured Hessian")
plt.title("Geometry changes while computation stays fixed")
plt.legend()
plt.grid(True, which="both")

fig_path = FIGURES / "29_geometry_changes.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 4. What survives?

This is the central checklist.

Equivalent representations preserve the computation but can alter geometric measurements.

In [ ]:
invariants = pd.DataFrame([
    {"item": "Function", "survives": True, "category": "computation"},
    {"item": "Predictions", "survives": True, "category": "computation"},
    {"item": "Test error", "survives": True, "category": "evaluation"},
    {"item": "Effective input-output behavior", "survives": True, "category": "computation"},
    {"item": "Hessian sharpness", "survives": False, "category": "geometry"},
    {"item": "Parameter coordinates", "survives": False, "category": "representation"},
    {"item": "Parameter norm", "survives": False, "category": "representation"},
    {"item": "Coordinate-space curvature", "survives": False, "category": "geometry"},
])

invariants

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axis("off")

labels = []
for _, row in invariants.iterrows():
    mark = "✓" if row["survives"] else "✗"
    labels.append([mark, row["item"], row["category"]])

table = ax.table(
    cellText=labels,
    colLabels=["Survives?", "Property", "Category"],
    cellLoc="left",
    colLoc="left",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 1.7)

ax.set_title("Computation survives. Geometry may not.", pad=18)

fig_path = FIGURES / "29_invariants.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 5. Connection to linear representation

This repo connects naturally to linear representation work.

Both questions ask:

> Which geometric facts are intrinsic to the computation, and which depend on the chosen representation?

For flat minima:

- Hessian geometry can change.
- Sharpness can change.
- The function can remain fixed.

For linear representation:

- Directions can appear meaningful.
- Coordinate choices can shape interpretation.
- Mechanistic explanations require invariants.

## 6. Engineering statement

> Some properties belong to the computation.
>
> Others belong to the representation.
>
> Scientific explanations should distinguish between them.

In [ ]:
summary_csv = RESULTS / "29_representation_summary.csv"
summary_json = RESULTS / "29_representation_summary.json"

combined = properties.copy()
combined.to_csv(summary_csv, index=False)

summary = {
    "title": "Representation Matters",
    "same_computation": True,
    "geometry_changes": True,
    "max_sharpness_ratio": float(geometry["sharpness_ratio"].max()),
    "min_sharpness_ratio": float(geometry["sharpness_ratio"].min()),
    "engineering_statement": (
        "Some properties belong to the computation. "
        "Others belong to the representation. "
        "Scientific explanations should distinguish between them."
    )
}

with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", summary_csv)
print("Saved:", summary_json)
print(summary)

In [ ]:
output_files = [
    FIGURES / "29_property_table.png",
    FIGURES / "29_same_computation.png",
    FIGURES / "29_geometry_changes.png",
    FIGURES / "29_invariants.png",
    RESULTS / "29_representation_summary.csv",
    RESULTS / "29_representation_summary.json",
]

zip_path = DOWNLOADS / "29_representation_matters_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if f.exists():
            z.write(f, arcname=f.relative_to(ROOT))

print("Created:", zip_path)
print("\nIncluded files:")
for f in output_files:
    print("-", f.relative_to(ROOT), "exists=" + str(f.exists()))

## Download outputs

In Colab, run the next cell to download the ZIP directly.

In local Jupyter, open:

`downloads/29_representation_matters_outputs.zip`

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
    for f in output_files:
        if f.exists():
            display(FileLink(str(f)))